In [17]:
# FILE: 00_Python_Raw_Data_Load.ipynb
# PURPOSE:
# Load all CSV source files from local folder into PostgreSQL raw schema.
#
# PROJECT:
# Loan and Credit Card Risk Analysis using Python and PostgreSQL
#
# TARGET SCHEMA:
# raw
#
# LOGIC:
# 1. Connect to PostgreSQL
# 2. Create raw schema if it does not exist
# 3. Read all CSV files from source folder
# 4. Create corresponding raw tables with TEXT columns
# 5. Truncate existing data for reload
# 6. Bulk load CSV data using COPY
# ============================================================

import os
import pandas as pd
import psycopg2

In [18]:
# STEP 1: Define Source Data Folder
# ============================================================

folder_path = r"D:\16_Projects\Retail_Loan_Portfolio_Synthetic_Data\Data"

In [19]:
# STEP 2: Connect to PostgreSQL database
# ============================================================

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="Retail Loan Portfolio & Customer Compliance Analytics",
    user="postgres",
    password="Bhushan148"
)

cur = conn.cursor()

# Create raw schema if not exists
cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
conn.commit()

print("✅ Connected to PostgreSQL and raw schema is ready.")

✅ Connected to PostgreSQL and raw schema is ready.


In [20]:
# STEP 3: Identify all CSV files in source folder

csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]

print("✅ Total CSV files found:", len(csv_files))
print(csv_files)

✅ Total CSV files found: 6
['Branch.csv', 'Customer.csv', 'Geography.csv', 'KYC.csv', 'Loan.csv', 'Loan_Product.csv']


In [21]:
# STEP 4: Load each CSV file into raw schema
# LOGIC:
# - Table name = CSV file name without extension
# - All columns are created as TEXT in raw layer
# - Existing table data is truncated before reload
# - Data is bulk loaded using PostgreSQL COPY

for file in csv_files:
    file_path = os.path.join(folder_path, file)

    # Table name from file name
    table_name = file.replace(".csv", "").lower()

    # Read only header row to extract column names
    df_header = pd.read_csv(file_path, nrows=0)
    columns = df_header.columns.tolist()

    # Create raw table with all columns as TEXT
    cols_sql = ", ".join([f'"{col}" TEXT' for col in columns])
    create_table_sql = f'''
        CREATE TABLE IF NOT EXISTS raw."{table_name}" (
            {cols_sql}
        );
    '''
    cur.execute(create_table_sql)

    # Truncate existing table data before reloading
    cur.execute(f'TRUNCATE TABLE raw."{table_name}";')

    # COPY command for fast bulk load
    copy_sql = f'''
        COPY raw."{table_name}" ({", ".join([f'"{col}"' for col in columns])})
        FROM STDIN
        WITH (FORMAT csv, HEADER true)
    '''

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        cur.copy_expert(copy_sql, f)

    conn.commit()
    print(f"✅ Loaded file: {file}  -->  raw.{table_name}")

✅ Loaded file: Branch.csv  -->  raw.branch
✅ Loaded file: Customer.csv  -->  raw.customer
✅ Loaded file: Geography.csv  -->  raw.geography
✅ Loaded file: KYC.csv  -->  raw.kyc
✅ Loaded file: Loan.csv  -->  raw.loan
✅ Loaded file: Loan_Product.csv  -->  raw.loan_product


In [22]:
# STEP 5: Verify Loaded RAW Tables
# ============================================================

verification_sql = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'raw'
ORDER BY table_name;
"""

cur.execute(verification_sql)
raw_tables = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
raw_tables

,table_schema,table_name
0,raw,branch
1,raw,customer
2,raw,geography
3,raw,kyc
4,raw,loan
5,raw,loan_product


In [23]:
# STEP 6: Close database connection

cur.close()
conn.close()

print("🔌 PostgreSQL connection closed successfully.")

🔌 PostgreSQL connection closed successfully.
